# DataFrame Transformations: Select, Filter, WithColumn & Expressions

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 03_dataframe_transformations.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Complete practical guide to DataFrame transformations: select, selectExpr, filter/where, withColumn, withColumnRenamed, drop, distinct, and column functions.
- **Why does this topic exist?**: Data engineers spend 80% of their coding time transforming columns, cleaning text, and manipulating dates and timestamps.
- **What problem does it solve?**: Suboptimal expressions (like chained withColumn in tight loops) degrade Catalyst optimizer performance.
- **Where is it used in real-world Data Engineering?**: Data wrangling, feature engineering, and data cleaning pipelines.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | DataFrame fundamentals, schemas |
| **Required depth** | Working knowledge: can build clean, readable transformation pipelines |
| **Data Engineer** | Write chained transformations with built-in functions and inspect the resulting plan. |
| **Senior Data Engineer** | Choose expressions that enable pushdown/pruning and avoid accidental shuffles. |
| **Spark specialist** | Reason about column lineage, codegen, and optimizer rules. |
| **Next topic** | `03_dataframe_and_spark_sql/04_spark_sql.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Most DataFrame code is a chain of small transformations. The danger is that a convenient expression can
block optimization or add a hidden shuffle. For example, a Python UDF prevents codegen, and a filter applied after
an aggregation cannot reduce the input rows. Good transformation design keeps filters and projections early.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Projection** | Choose columns | `select`, `drop`, `withColumn` |
| **Selection** | Choose rows | `filter`/`where`, `limit` |
| **Expression** | A computed column | `F.col`, `F.when`, arithmetic, functions |
| **Immutability** | Each operation returns a new DataFrame | No in-place mutation |
| **Chaining** | Compose transformations | Readable pipelines |
| **Pushdown** | Move work closer to the source | Filter/column pruning |

### How it works, step by step

1. Each transformation adds a node to the logical plan.
2. Catalyst combines and reorders compatible operators.
3. Projection pruning removes unused columns from the scan.
4. Predicate pushdown moves filters toward the source when possible.
5. Wide transformations add an exchange and a stage boundary.
6. The physical plan decides whether codegen can fuse operators.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Core Transformation Categories

```
+---------------------------------------------------------------+
| DATAFRAME TRANSFORMATIONS                                     |
+---------------------------------------------------------------+
| Structural:                                                   |
|  - select(), selectExpr(), withColumn(), withColumnRenamed()  |
|  - drop(), dropDuplicates()                                   |
| Row-Level Filtering:                                          |
|  - filter(), where()                                          |
| Column-Level Expressions:                                     |
|  - col(), lit(), when().otherwise()                           |
|  - String: lower, upper, trim, regexp_replace, split          |
|  - Date: to_date, to_timestamp, datediff, date_add, year      |
|  - Numeric: round, ceil, floor, abs                           |
+---------------------------------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Transformations_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: End-to-End Data Transformation Pipeline

We transform `/opt/spark-data/retail/orders.csv` with realistic business logic.

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")

# Chained transformation pipeline
transformed = (
    orders
    .filter(F.col("status").isin("Completed", "Shipped"))
    .withColumn("order_date", F.to_date(F.col("order_ts")))
    .withColumn("order_year", F.year(F.col("order_date")))
    .withColumn("order_month", F.month(F.col("order_date")))
    .withColumn("gross_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("discount_amount", F.round(F.col("gross_amount") * F.col("discount"), 2))
    .withColumn("net_amount", F.round(F.col("gross_amount") - F.col("discount_amount"), 2))
    .withColumn("channel_category", F.when(F.col("channel") == "Store", "Physical").otherwise("Digital"))
    .select(
        "order_id", "customer_id", "order_date", "order_year", "order_month",
        "gross_amount", "discount_amount", "net_amount", "channel_category", "status"
    )
)

transformed.show(5)

## 4. High-Performance Column Replacement with `select` vs Multiple `withColumn`

A common anti-pattern is calling `.withColumn()` 50 times in a row, which bloats the Catalyst tree. Using `select` or `selectExpr` generates a cleaner plan.

In [ ]:
# High performance multi-column projection
clean_df = orders.select(
    F.col("order_id"),
    F.col("customer_id"),
    F.col("quantity"),
    F.col("unit_price"),
    (F.col("quantity") * F.col("unit_price")).alias("raw_total"),
    F.upper(F.col("channel")).alias("channel_upper")
)
clean_df.show(3)

---

## Experiment: Chained withColumn vs Projection Plan Complexity

- **1. What are we testing?**: Compare the analyzed logical plan tree depth between 10 chained withColumn calls vs a single select projection.
- **2. What do we expect?**: 10 withColumns creates 10 nested Project operators; a single select creates exactly 1 Project operator.

In [ ]:
# 10 chained withColumns
df_chained = orders
for i in range(10):
    df_chained = df_chained.withColumn(f"col_{i}", F.lit(i))

# Single select
df_single = orders.select("*", *[F.lit(i).alias(f"col_{i}") for i in range(10)])

print(f"Chained withColumn Plan Tree Depth: {len(df_chained._jdf.queryExecution().analyzed().treeString().splitlines())}")
print(f"Single Select Plan Tree Depth:      {len(df_single._jdf.queryExecution().analyzed().treeString().splitlines())}")

**3. What happened?**: Chained withColumns generated a massive nested logical plan tree.  
**4. Why did it happen?**: Each withColumn creates a new DataFrame wrapper node in the unresolved logical tree.  
**5. Practical Engineering Lesson**: When adding or transforming multiple columns, prefer a single select() or selectExpr() call.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Customer 360 feature engineering

- **Industry**: Telecommunications
- **Scenario**: A telecom team builds a customer feature table from usage, billing, and support data.
- **Problem**: Hundreds of derived columns make the pipeline hard to read and easy to break.
- **Why this concept is relevant**: Small, named transformation steps with built-in functions are easier to test and let Catalyst prune unused columns.
- **Implementation idea**: Create a typed base DataFrame, add feature groups in separate functions, select the final feature list, and write Parquet.
- **Expected result**: A maintainable feature table with a readable plan.
- **Alternative**: A feature store with managed transformations.
- **Trade-offs**: Spark is flexible for batch feature engineering; a feature store adds discovery, versioning, and serving.

### Use Case 2: Cleaning a marketing clickstream

- **Industry**: Advertising
- **Scenario**: A marketing team cleans bot traffic, normalizes URLs, and derives campaign fields.
- **Problem**: The raw data has inconsistent casing, nulls, and malformed timestamps.
- **Why this concept is relevant**: Column expressions and `when`/`coalesce` handle the cleaning declaratively without Python UDFs.
- **Implementation idea**: Filter invalid events, normalize with built-ins, derive fields, and validate counts before and after.
- **Expected result**: A clean clickstream with an auditable transformation chain.
- **Alternative**: A streaming cleanup job in Kafka Streams or Flink.
- **Trade-offs**: Batch Spark is simple for historical cleanup; streaming is better for live dashboards.

---

## Business Value

- Cleaner pipelines reduce maintenance cost.
- Early filters/projections reduce I/O and shuffle cost.
- Built-in functions avoid Python serialization overhead.
- Readable transformations make code review and debugging faster.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Filter and select before joins and aggregations.
- Prefer built-in functions over UDFs.
- Avoid repeated `withColumn` calls that create deeply nested plans; use `select` with expressions.
- Check column ambiguity after joins and qualify names.
- Inspect `explain()` for pushdown and pruning.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Filter placement

**Steps**

1. Build a query that joins orders and products, then filters by category.
2. Move the filter before the join and compare the plans.
3. Check the SQL tab for rows read and shuffled.

**What to observe**: Filtering the dimension before the join can reduce shuffle and scan work.

**Healthy result**: The filter appears close to the scan in the physical plan.

**Problematic result**: A filter after a wide join may not reduce the join input.

**Engineering takeaway**: Put filters where they reduce the most data.

---

## Trade-offs and Alternatives

### Advantages

- Readable and composable
- Optimizer-friendly
- Works with SQL

### Limitations

- Easy to create hidden shuffles
- Complex chains can be hard to read
- UDFs break optimization

### When to use it

- ETL
- Feature engineering
- Data cleaning

### When not to use it

- Row-by-row Python
- Repeated wide transformations
- Unqualified columns after joins

### Alternatives inside Spark

- Spark SQL
- `selectExpr`
- Built-in functions

### Alternatives outside Spark

- pandas
- dbt
- SQL in a warehouse

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Rename and Cast Columns
**Problem Statement**: Rename'quantity'to'units_ordered'and cast'unit_price'to DecimalType(10, 2).  
**Input / Context**: orders DataFrame

In [ ]:
# Exercise 1: Rename and cast

### Exercise 2 (Intermediate): Extract Domain from Customer Email
**Problem Statement**: Load customers.csv and use regexp_extract or split to extract email domains into a'domain'column.  
**Input / Context**: /opt/spark-data/retail/customers.csv

In [ ]:
# Exercise 2: Email domain extraction

### Exercise 3 (Advanced): Conditional Multi-Tier Discount Calculation
**Problem Statement**: Calculate a loyalty discount percentage: Gold = 20%, Silver = 10%, Bronze/Consumer = 0% using nested when().otherwise().  
**Input / Context**: custs DataFrame

In [ ]:
# Exercise 3: Loyalty discount tier

---

## Production Debugging Exercise

### Ambiguous column after join

**Scenario**: After joining orders and customers, `customer_id` is ambiguous and the code fails.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Qualify columns with aliases.
2. Drop duplicate key columns or use a join condition.
3. Inspect the schema after the join.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Rename and Cast Columns):
```python
ex1 = orders.withColumnRenamed("quantity","units_ordered").withColumn("unit_price", F.col("unit_price").cast("decimal(10,2)"))
ex1.printSchema()
```

#### Solution 2 (Extract Domain from Customer Email):
```python
custs = spark.read.option("header", True).csv("/opt/spark-data/retail/customers.csv")
custs_domain = custs.withColumn("domain", F.split(F.col("email"),"@").getItem(1))
custs_domain.select("email","domain").show(5)
```

#### Solution 3 (Conditional Multi-Tier Discount Calculation):
```python
discount_df = custs.withColumn("tier_discount",
    F.when(F.col("loyalty_tier") =="Gold", 0.20)
     .when(F.col("loyalty_tier") =="Silver", 0.10)
     .otherwise(0.0)
)
discount_df.select("first_name","loyalty_tier","tier_discount").show(5)
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Vectorized Transformations**: Always use built-in pyspark.sql.functions for string, date, and math operations.
- **Single Projection**: Use select() with expressions rather than dozens of chained withColumns.
- **Column Expressions**: Columns can be dynamically created with F.col() and F.lit().

### Essential Technical Terms
- `selectExpr`: Select projection accepting raw SQL string expressions.
- `when().otherwise()`: Conditional case-when statement.
- `cast()`: Converts a column to another DataType.

### Common Engineering Mistakes
- **50 withColumn calls in a loop**: Causes excessive plan complexity and slow analysis.
- **Missing lit() for constants**: Passing raw Python values into col() throws type errors.

### Interview Questions on This Topic
- **Q: What is the difference between filter() and where()?**
  - *A*: They are exact aliases of each other.
- **Q: Why is select() preferred over multiple withColumns?**
  - *A*: Creates a single Project operator in the logical plan rather than nesting dozens of intermediate projections.


### More Interview Questions

- **Q: What is the difference between `select` and `withColumn`?**
  - *A*: `select` chooses/projects columns and expressions; `withColumn` adds or replaces one column and returns a new DataFrame.
- **Q: When should you avoid a Python UDF?**
  - *A*: When a built-in function can express the logic, because UDFs add serialization and prevent whole-stage codegen.

### What to Study Next
Open 03_dataframe_and_spark_sql/04_spark_sql.ipynb to master SQL queries and temporary views.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')